<a href="https://colab.research.google.com/github/yogurt-yona/COS3109/blob/main/Informedsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# ข้อ 1. Table of Content & System Architecture Definition
# =========================================================

import heapq

# 1.1 Problem Definition Layer (คลาสฐานนิยามโจทย์)
class Search_problem(object):
    def start_node(self):
        raise NotImplementedError("start_node")
    def is_goal(self, node):
        raise NotImplementedError("is_goal")
    def neighbors(self, node):
        raise NotImplementedError("neighbors")
    def heuristic(self, n):
        return 0

# 1.2 Arc Class (คลาสเก็บข้อมูลเส้นเชื่อมและ Cost)
class Arc(object):
    def __init__(self, from_node, to_node, cost=1, action=None):
        assert cost >= 0, ("Cost cannot be negative for " + str(from_node) + "->" + str(to_node) + ", cost: " + str(cost))
        self.from_node = from_node
        self.to_node = to_node
        self.action = action
        self.cost = cost

    def __repr__(self):
        if self.action:
            return str(self.from_node) + " --" + str(self.action) + "--> " + str(self.to_node)
        else:
            return str(self.from_node) + " --> " + str(self.to_node)

# 1.3 Explicit Graph Search Problem Layer
class Search_problem_from_explicit_graph(Search_problem):
    def __init__(self, nodes, arcs, start=None, goals=set(), hmap={}):
        self.neighs = {}
        self.nodes = nodes
        for node in nodes:
            self.neighs[node] = []
        self.arcs = arcs
        for arc in arcs:
            self.neighs[arc.from_node].append(arc)
        self.start = start
        self.goals = goals
        self.hmap = hmap

    def start_node(self):
        return self.start

    def is_goal(self, node):
        return node in self.goals

    def neighbors(self, node):
        return self.neighs[node]

    def heuristic(self, node):
        return self.hmap.get(node, 0)

# 1.4 Path Representation Layer (คลาสจัดการเส้นทาง)
class Path(object):
    def __init__(self, initial, arc=None):
        self.initial = initial
        self.arc = arc
        if arc is None:
            self.cost = 0
        else:
            self.cost = initial.cost + arc.cost

    def end(self):
        if self.arc is None:
            return self.initial
        else:
            return self.arc.to_node

    def nodes(self):
        current = self
        while current.arc is not None:
            yield current.arc.to_node
            current = current.initial
        yield current.initial

    def __repr__(self):
        if self.arc is None:
            return str(self.initial)
        elif self.arc.action:
            return str(self.initial) + "\n   --" + str(self.arc.action) + "--> " + str(self.arc.to_node)
        else:
            return str(self.initial) + " --> " + str(self.arc.to_node)

# 1.5 Displayable Helper Class
class Displayable(object):
    max_display_level = 1
    def display(self, level, *args, **nargs):
        if level <= self.max_display_level:
            print(*args, **nargs)

print("✅ ข้อ 1: โหลดคลาสสถาปัตยกรรมระบบค้นหาเรียบร้อยแล้ว")

✅ ข้อ 1: โหลดคลาสสถาปัตยกรรมระบบค้นหาเรียบร้อยแล้ว


In [5]:
# =========================================================
# ข้อ 2. นิยามกราฟและสร้าง Tree Structure สำหรับ Problem 1 & 2
# =========================================================

# นิยาม Problem 1
problem1 = Search_problem_from_explicit_graph(
    {'a', 'b', 'c', 'd', 'g'},
    [Arc('a', 'b', 1), Arc('a', 'c', 3), Arc('b', 'd', 3), Arc('b', 'c', 1),
     Arc('c', 'd', 1), Arc('c', 'g', 3), Arc('d', 'g', 1)],
    start='a',
    goals={'g'}
)

# นิยาม Problem 2
problem2 = Search_problem_from_explicit_graph(
    {'a', 'b', 'c', 'd', 'e', 'g', 'h', 'j'},
    [Arc('a', 'b', 1), Arc('b', 'c', 3), Arc('b', 'd', 1), Arc('d', 'e', 3),
     Arc('d', 'g', 1), Arc('a', 'h', 3), Arc('h', 'j', 1)],
    start='a',
    goals={'g'}
)

def print_tree_structure(problem, problem_name):
    print(f"\n==================== Search Tree for {problem_name} ====================")
    print(f"Start Node: [{problem.start_node()}] | Goal Nodes: {problem.goals}")
    print("-" * 65)

    def expand_node(node, indent=0, current_cost=0):
        prefix = "  " * indent + "└─ " if indent > 0 else ""
        is_goal_str = " (GOAL ★)" if problem.is_goal(node) else ""
        print(f"{prefix}Node [{node}] (g={current_cost}){is_goal_str}")

        for arc in problem.neighbors(node):
            expand_node(arc.to_node, indent + 1, current_cost + arc.cost)

    expand_node(problem.start_node())

# พิมพ์แสดง Search Tree ของทั้งสองปัญหา
print_tree_structure(problem1, "Problem 1")
print_tree_structure(problem2, "Problem 2")


==================== Search Tree for Problem 1 ====================
Start Node: [a] | Goal Nodes: {'g'}
-----------------------------------------------------------------
Node [a] (g=0)
  └─ Node [b] (g=1)
    └─ Node [d] (g=4)
      └─ Node [g] (g=5) (GOAL ★)
    └─ Node [c] (g=2)
      └─ Node [d] (g=3)
        └─ Node [g] (g=4) (GOAL ★)
      └─ Node [g] (g=5) (GOAL ★)
  └─ Node [c] (g=3)
    └─ Node [d] (g=4)
      └─ Node [g] (g=5) (GOAL ★)
    └─ Node [g] (g=6) (GOAL ★)

==================== Search Tree for Problem 2 ====================
Start Node: [a] | Goal Nodes: {'g'}
-----------------------------------------------------------------
Node [a] (g=0)
  └─ Node [b] (g=1)
    └─ Node [c] (g=4)
    └─ Node [d] (g=2)
      └─ Node [e] (g=5)
      └─ Node [g] (g=3) (GOAL ★)
  └─ Node [h] (g=3)
    └─ Node [j] (g=4)


In [7]:
# =========================================================
# ข้อ 3. Explicit Graph & Complete Search Tree สำหรับ Problem 2
# =========================================================

# 1. นิยามโครงสร้างชั้นข้อมูล
class Search_problem(object):
    def start_node(self): raise NotImplementedError("start_node")
    def is_goal(self, node): raise NotImplementedError("is_goal")
    def neighbors(self, node): raise NotImplementedError("neighbors")

class Arc(object):
    def __init__(self, from_node, to_node, cost=1, action=None):
        self.from_node = from_node
        self.to_node = to_node
        self.action = action
        self.cost = cost

class Search_problem_from_explicit_graph(Search_problem):
    def __init__(self, nodes, arcs, start=None, goals=set(), hmap={}):
        self.neighs = {node: [] for node in nodes}
        self.nodes = nodes
        self.arcs = arcs
        for arc in arcs:
            self.neighs[arc.from_node].append(arc)
        self.start = start
        self.goals = goals
        self.hmap = hmap

    def start_node(self): return self.start
    def is_goal(self, node): return node in self.goals
    def neighbors(self, node): return self.neighs[node]

# 2. นิยามกราฟ Problem 2
problem2 = Search_problem_from_explicit_graph(
    {'a', 'b', 'c', 'd', 'e', 'g', 'h', 'j'},
    [Arc('a', 'b', 1), Arc('b', 'c', 3), Arc('b', 'd', 1), Arc('d', 'e', 3),
     Arc('d', 'g', 1), Arc('a', 'h', 3), Arc('h', 'j', 1)],
    start='a',
    goals={'g'}
)

# 3.1 แสดงผล Explicit Graph Arcs
print("==================== 3.1 Explicit Graph Structure (Problem 2) ====================")
print("โหนดและเส้นเชื่อม (Arcs & Costs) ทั้งหมดใน Problem 2:")
for arc in problem2.arcs:
    print(f"  • Node [{arc.from_node}] --(cost: {arc.cost})--> Node [{arc.to_node}]")

# 3.2 แสดงผล Complete Search Tree (การขยายกิ่งครบทุกระดับ)
print("\n==================== 3.2 Complete Search Tree (Problem 2) ====================")
def print_complete_tree(node, indent=0, current_cost=0):
    prefix = "  " * indent + "└─ " if indent > 0 else ""
    is_goal_str = " (GOAL ★)" if problem2.is_goal(node) else ""
    print(f"{prefix}Node [{node}] (g={current_cost}){is_goal_str}")

    # ขยายกิ่งเพื่อนบ้านทั้งหมด (Complete Tree Expansion)
    for arc in problem2.neighbors(node):
        print_complete_tree(arc.to_node, indent + 1, current_cost + arc.cost)

print_complete_tree(problem2.start_node())

==================== 3.1 Explicit Graph Structure (Problem 2) ====================
โหนดและเส้นเชื่อม (Arcs & Costs) ทั้งหมดใน Problem 2:
  • Node [a] --(cost: 1)--> Node [b]
  • Node [b] --(cost: 3)--> Node [c]
  • Node [b] --(cost: 1)--> Node [d]
  • Node [d] --(cost: 3)--> Node [e]
  • Node [d] --(cost: 1)--> Node [g]
  • Node [a] --(cost: 3)--> Node [h]
  • Node [h] --(cost: 1)--> Node [j]

==================== 3.2 Complete Search Tree (Problem 2) ====================
Node [a] (g=0)
  └─ Node [b] (g=1)
    └─ Node [c] (g=4)
    └─ Node [d] (g=2)
      └─ Node [e] (g=5)
      └─ Node [g] (g=3) (GOAL ★)
  └─ Node [h] (g=3)
    └─ Node [j] (g=4)


In [9]:
# =========================================================
# ข้อ 4. Graphical Solution สำหรับ DFS และ A* (ฉบับแก้ไข Error)
# =========================================================

import heapq

# 1. Base Class Search Problem
class Search_problem(object):
    def start_node(self): raise NotImplementedError("start_node")
    def is_goal(self, node): raise NotImplementedError("is_goal")
    def neighbors(self, node): raise NotImplementedError("neighbors")
    def heuristic(self, n): return 0

# 2. Arc Class
class Arc(object):
    def __init__(self, from_node, to_node, cost=1, action=None):
        self.from_node = from_node
        self.to_node = to_node
        self.action = action
        self.cost = cost

    def __repr__(self):
        return (str(self.from_node) + " --" + str(self.action) + "--> " + str(self.to_node)) if self.action else (str(self.from_node) + " --> " + str(self.to_node))

# 3. Explicit Graph Class (เพิ่มฟังก์ชัน heuristic ป้องกัน AttributeError)
class Search_problem_from_explicit_graph(Search_problem):
    def __init__(self, nodes, arcs, start=None, goals=set(), hmap={}):
        self.neighs = {node: [] for node in nodes}
        self.nodes = nodes
        self.arcs = arcs
        for arc in arcs:
            self.neighs[arc.from_node].append(arc)
        self.start = start
        self.goals = goals
        self.hmap = hmap

    def start_node(self): return self.start
    def is_goal(self, node): return node in self.goals
    def neighbors(self, node): return self.neighs[node]

    # *** ฟังก์ชันแก้ไขจุด Error ***
    def heuristic(self, node):
        return self.hmap.get(node, 0)

# 4. Path Class
class Path(object):
    def __init__(self, initial, arc=None):
        self.initial = initial
        self.arc = arc
        self.cost = 0 if arc is None else initial.cost + arc.cost

    def end(self): return self.initial if self.arc is None else self.arc.to_node

    def __repr__(self):
        if self.arc is None: return str(self.initial)
        return (str(self.initial) + "\n   --" + str(self.arc.action) + "--> " + str(self.arc.to_node)) if self.arc.action else (str(self.initial) + " --> " + str(self.arc.to_node))

# 5. Displayable Helper Class
class Displayable(object):
    max_display_level = 1
    def display(self, level, *args, **nargs):
        if level <= self.max_display_level: print(*args, **nargs)

# 6. DFS Searcher Algorithm
class Searcher(Displayable):
    def __init__(self, problem):
        self.problem = problem
        self.initialize_frontier()
        self.num_expanded = 0
        self.add_to_frontier(Path(problem.start_node()))
        super().__init__()

    def initialize_frontier(self): self.frontier = []
    def empty_frontier(self): return self.frontier == []
    def add_to_frontier(self, path): self.frontier.append(path)

    def search(self):
        while not self.empty_frontier():
            path = self.frontier.pop()
            self.num_expanded += 1
            if self.problem.is_goal(path.end()):
                self.solution = path
                return path
            else:
                for arc in reversed(self.problem.neighbors(path.end())):
                    self.add_to_frontier(Path(path, arc))
        return None

# 7. Priority Queue for A* Search
class FrontierPQ(object):
    def __init__(self):
        self.frontier_index = 0
        self.frontierpq = []

    def empty(self): return self.frontierpq == []
    def add(self, path, value):
        self.frontier_index += 1
        heapq.heappush(self.frontierpq, (value, -self.frontier_index, path))
    def pop(self):
        (_, _, path) = heapq.heappop(self.frontierpq)
        return path

# 8. A* Searcher Algorithm
class AStarSearcher(Searcher):
    def initialize_frontier(self): self.frontier = FrontierPQ()
    def empty_frontier(self): return self.frontier.empty()
    def add_to_frontier(self, path):
        value = path.cost + self.problem.heuristic(path.end())
        self.frontier.add(path, value)

# =========================================================
# DEFINING PROBLEMS & RUNNING SOLUTIONS
# =========================================================

problem1 = Search_problem_from_explicit_graph(
    {'a', 'b', 'c', 'd', 'g'},
    [Arc('a', 'b', 1), Arc('a', 'c', 3), Arc('b', 'd', 3), Arc('b', 'c', 1),
     Arc('c', 'd', 1), Arc('c', 'g', 3), Arc('d', 'g', 1)],
    start='a',
    goals={'g'}
)

problem2 = Search_problem_from_explicit_graph(
    {'a', 'b', 'c', 'd', 'e', 'g', 'h', 'j'},
    [Arc('a', 'b', 1), Arc('b', 'c', 3), Arc('b', 'd', 1), Arc('d', 'e', 3),
     Arc('d', 'g', 1), Arc('a', 'h', 3), Arc('h', 'j', 1)],
    start='a',
    goals={'g'}
)

def run_graphical_solutions():
    print("==================== 4.1 Solution Results for Problem 1 ====================")
    s1_dfs = Searcher(problem1)
    p1_dfs = s1_dfs.search()
    print(f"1) DFS Solution Path : {p1_dfs} (Total Cost: {p1_dfs.cost})")
    print(f"   [Expanded {s1_dfs.num_expanded} paths]")

    s1_astar = AStarSearcher(problem1)
    p1_astar = s1_astar.search()
    print(f"2) A* Optimal Path  : {p1_astar} (Total Cost: {p1_astar.cost})")
    print(f"   [Expanded {s1_astar.num_expanded} paths]")

    print("\n==================== 4.2 Solution Results for Problem 2 ====================")
    s2_dfs = Searcher(problem2)
    p2_dfs = s2_dfs.search()
    print(f"1) DFS Solution Path : {p2_dfs} (Total Cost: {p2_dfs.cost})")
    print(f"   [Expanded {s2_dfs.num_expanded} paths]")

    s2_astar = AStarSearcher(problem2)
    p2_astar = s2_astar.search()
    print(f"2) A* Optimal Path  : {p2_astar} (Total Cost: {p2_astar.cost})")
    print(f"   [Expanded {s2_astar.num_expanded} paths]")

run_graphical_solutions()

==================== 4.1 Solution Results for Problem 1 ====================
1) DFS Solution Path : a --> b --> d --> g (Total Cost: 5)
   [Expanded 4 paths]
2) A* Optimal Path  : a --> b --> c --> d --> g (Total Cost: 4)
   [Expanded 7 paths]

==================== 4.2 Solution Results for Problem 2 ====================
1) DFS Solution Path : a --> b --> d --> g (Total Cost: 3)
   [Expanded 6 paths]
2) A* Optimal Path  : a --> b --> d --> g (Total Cost: 3)
   [Expanded 4 paths]
